In [1]:
from datasets import load_dataset

In [2]:
dataset = load_dataset("roneneldan/TinyStories", split= "train[:1%]")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [3]:
## To load the full dataset

# for split in dataset:
#     output_path = f"{split}.txt"
#     with open(output_path, "w", encoding="utf-8") as f:
#         for story in dataset[split]["text"]:
#             f.write(story.strip() + "\n\n")

#     print(f"Saved {split} split to {output_path}")

## To load limited part of the dataset

with open("train.txt", "w", encoding="utf-8") as f:
    for story in dataset["text"]:
        f.write(story.strip() + "\n\n")

In [ ]:
with open("train.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [ ]:
print("Dataset size:", len(text))

Dataset size: 18995021


In [ ]:
print(text[:500])

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them b


In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))
print(vocab_size)


 !"$&'()*+,-./0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz ¡¦©«±³»ÂÃâœ˜“”€™
97


In [ ]:
stoi = {ch: i for i,ch in enumerate(chars)}
itos = {i: ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)

print(data.shape, data.dtype)
print(data[:100])

torch.Size([18995021]) torch.int64
tensor([42, 67, 58,  1, 57, 54, 78, 11,  1, 54,  1, 65, 62, 73, 73, 65, 58,  1,
        60, 62, 71, 65,  1, 67, 54, 66, 58, 57,  1, 39, 62, 65, 78,  1, 59, 68,
        74, 67, 57,  1, 54,  1, 67, 58, 58, 57, 65, 58,  1, 62, 67,  1, 61, 58,
        71,  1, 71, 68, 68, 66, 13,  1, 46, 61, 58,  1, 64, 67, 58, 76,  1, 62,
        73,  1, 76, 54, 72,  1, 57, 62, 59, 59, 62, 56, 74, 65, 73,  1, 73, 68,
         1, 69, 65, 54, 78,  1, 76, 62, 73, 61])


In [ ]:
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

In [ ]:
# block_size = 8
# # captures context of characters of certain length
# train_data[:block_size + 1] # plus one is used in chunking

In [ ]:
torch.manual_seed(1337)

batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1: i + block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch("train")
print("inputs")
print(xb.shape)
print(xb)
print("targets")
print(yb.shape)
print(yb)

inputs
torch.Size([4, 8])
tensor([[67, 73, 74, 54, 65, 65, 78,  1],
        [35, 58,  1, 76, 54, 67, 73, 58],
        [65, 62, 66, 55, 58, 57,  1, 73],
        [57,  1, 54, 73,  1, 73, 61, 58]], device='cuda:0')
targets
torch.Size([4, 8])
tensor([[73, 74, 54, 65, 65, 78,  1, 61],
        [58,  1, 76, 54, 67, 73, 58, 57],
        [62, 66, 55, 58, 57,  1, 73, 61],
        [ 1, 54, 73,  1, 73, 61, 58,  1]], device='cuda:0')


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel(vocab_size)
m = model.to(device)

In [ ]:
eval_iters = 200

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
context = torch.zeros((1,1), dtype=torch.long, device = device)

In [ ]:
print(decode(model.generate(context, max_new_tokens=100)[0].tolist()))


aœvkimxQCKv;hN&hM'Kno”(mY'xMe8
U©impt;â/Mk.˜»±IL™gR?(*sn:4â3˜QBO
Sj+9GpxeS ;;fEbNœ:"4Mertv OIW)DgLmP


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [ ]:
max_iters = 10000
eval_interval = 300
batch_size = 32
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 5.0750, val loss 5.0756
step 300: train loss 4.6901, val loss 4.6907
step 600: train loss 4.3341, val loss 4.3362
step 900: train loss 4.0166, val loss 4.0152
step 1200: train loss 3.7482, val loss 3.7403
step 1500: train loss 3.4986, val loss 3.4978
step 1800: train loss 3.2983, val loss 3.2911
step 2100: train loss 3.1175, val loss 3.1285
step 2400: train loss 2.9821, val loss 2.9805
step 2700: train loss 2.8469, val loss 2.8452
step 3000: train loss 2.7403, val loss 2.7455
step 3300: train loss 2.6700, val loss 2.6599
step 3600: train loss 2.6004, val loss 2.5993
step 3900: train loss 2.5544, val loss 2.5569
step 4200: train loss 2.4987, val loss 2.5041
step 4500: train loss 2.4679, val loss 2.4723
step 4800: train loss 2.4479, val loss 2.4474
step 5100: train loss 2.4284, val loss 2.4186
step 5400: train loss 2.4065, val loss 2.4111
step 5700: train loss 2.3951, val loss 2.3907
step 6000: train loss 2.3740, val loss 2.3773
step 6300: train loss 2.3690, val loss 2

In [ ]:
decode(model.generate(context, max_new_tokens=100)[0].tolist())

'\n\n\nhelo r t pord Likikits therewoune uther busw. miry. tout s ne thed s bir sils534y, sa ed Itiming a'

### GPT


In [5]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('train.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = GPTLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))
#open('more.txt', 'w').write(decode(m.generate(context, max_new_tokens=10000)[0].tolist()))

10.813537 M parameters
step 0: train loss 4.7151, val loss 4.7165
step 500: train loss 1.3313, val loss 1.3165
step 1000: train loss 1.0246, val loss 1.0142
step 1500: train loss 0.9205, val loss 0.9107
step 2000: train loss 0.8528, val loss 0.8459
step 2500: train loss 0.8078, val loss 0.7992
step 3000: train loss 0.7743, val loss 0.7698
step 3500: train loss 0.7587, val loss 0.7522
step 4000: train loss 0.7398, val loss 0.7344
step 4500: train loss 0.7245, val loss 0.7197
step 4999: train loss 0.7110, val loss 0.7099

She held and take the flour. The flowers the three flood. She waited for it, please the skitch.

But there every time they eat out. But one at the apple and waited its flower. She watched the bug ynood.

Tom and Mia said, he wanted to shw the flowers too. They saw Bun and scratched the flood. They took the flower and break quickly close. They said grandma and safe nodding. They said to all the tasticase, sure soaved.

Bun smiled and said, "Yes, this. I did you spegar, m

In [6]:
print(decode(m.generate(context, max_new_tokens=15000)[0].tolist()))


She was brave and wondered what it was always tired.

She showed how it was something for their body and soft. She told something enjoyed melting with them. The mom was so proud of how it's inside better that's time dreaming as excitedling. With happiness, the boy and his softly said goodbye, and behavion and agded telling the mom all and he was competitive up.

But eventually, when the boy got too slow, Tinded in hand great. He proudly realized things to do and great. He knew that he was afraid it.

Tonde was real, so his mom urged her to be careful. Annd something to make him happy with and wrote the mouse, and remembered. When Daisy was tired, she saw a big, shunt was surprised, and threw the mouse on her mouth. She felt sad and nodded and cried.

When she acted a cookie, she saw also something small outside with a city smile. Pete did now mak surprises. He stood and put the mouse cried open its eyes. It spark and flew up the man to talk to the cords. And corder worked very fast an